# Fase 2: Experimentación de Modelos de Trading Intradía

## Objetivo
El objetivo de este notebook es experimentar con diferentes modelos de Machine Learning para predecir el precio futuro y generar señales de compra. Se comparará el rendimiento de varios modelos para determinar el más adecuado.

**Modelos a Probar:**
1.  **Modelo A (Baseline):** LightGBM - Un modelo de Gradient Boosting rápido y eficiente, bueno para datos tabulares.
2.  **Modelo B (Secuencial):** LSTM (Red Neuronal Recurrente) - Un modelo más complejo, diseñado para aprender de secuencias de datos.

### 1. Carga y Preparación de Datos

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import os

# Configurar rutas
DATA_DIR = '../data'
PROCESSED_DATA_PATH = os.path.join(DATA_DIR, 'processed_intraday_data.parquet')

# Cargar datos
df = pd.read_parquet(PROCESSED_DATA_PATH)

print(f"Datos cargados. Forma del DataFrame: {df.shape}")
print("Primeras 5 filas:")
df.head()

### 2. Definición de Variables Objetivo (Target)

Vamos a definir nuestros dos objetivos de predicción:

1.  **`target_price` (Regresión):** El precio que queremos predecir. Lo definiremos como el precio `N` pasos de tiempo en el futuro.
2.  **`target_signal` (Clasificación):** Una señal de 'COMPRA' (1) o 'NO COMPRAR' (0). La definiremos basada en si el precio futuro supera un cierto umbral de retorno.

In [ ]:
# Parámetros para la definición de targets
N_STEPS_FUTURE = 15  # Predecir el precio 15 'ticks' (puntos de tiempo) en el futuro
RETURN_THRESHOLD = 0.005 # 0.5% de retorno para considerar una señal de compra

# 1. Target de Regresión (Precio Futuro)
df['target_price'] = df['Price'].shift(-N_STEPS_FUTURE)

# 2. Target de Clasificación (Señal de Compra)
future_price = df['target_price']
current_price = df['Price']
returns = (future_price - current_price) / current_price

df['target_signal'] = (returns > RETURN_THRESHOLD).astype(int)

# Eliminar las últimas filas donde los targets son NaN (porque no hay datos futuros)
df.dropna(inplace=True)

print(f"Targets creados. Nueva forma del DataFrame: {df.shape}")
print("
Distribución de la señal de compra:")
print(df['target_signal'].value_counts(normalize=True))

### 3. Modelo A: LightGBM para Predicción de Precio (Regresión)

In [ ]:
# Seleccionar features y target
# Por ahora, usamos features simples. En una iteración futura, se pueden añadir más.
features = ['Price', 'Open_Daily', 'High_Daily', 'Low_Daily', 'Close_Daily', 'Volume_Daily', 'SMA_10',
            'C5_Open', 'C5_High', 'C5_Low', 'C5_Close', 'C5_Volume', 'Internal_Market_Price']
target_regression = 'target_price'

X = df[features]
y = df[target_regression]

# Dividir los datos en entrenamiento y prueba usando un split temporal
# Usaremos el 80% más antiguo para entrenar y el 20% más reciente para probar
split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f"Datos de entrenamiento: {X_train.shape[0]} filas")
print(f"Datos de prueba: {X_test.shape[0]} filas")

In [ ]:
# Entrenar el modelo de regresión LightGBM
print("Entrenando modelo LightGBM...")

lgbm_regressor = lgb.LGBMRegressor(
    objective='regression_l1', # MAE
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    n_jobs=-1,
    seed=42
)

lgbm_regressor.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='mae',
    callbacks=[lgb.early_stopping(100, verbose=True)]
)

print("
Entrenamiento completado.")

### 4. Evaluación del Modelo de Regresión

In [ ]:
# Realizar predicciones en el conjunto de prueba
y_pred = lgbm_regressor.predict(X_test)

# Calcular métricas de error
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

In [ ]:
# Visualizar los resultados
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(15, 7))

# Tomar una muestra de los datos para que el gráfico sea legible
sample_size = 2000
y_test_sample = y_test.iloc[-sample_size:]
y_pred_sample = y_pred[-sample_size:]

ax.plot(y_test_sample.index, y_test_sample, label='Precio Real', color='royalblue', lw=2)
ax.plot(y_test_sample.index, y_pred_sample, label='Precio Predicho', color='darkorange', lw=2, linestyle='--')

ax.set_title('Comparación de Precio Real vs. Predicho (Muestra de Prueba)', fontsize=16)
ax.set_xlabel('Fecha y Hora', fontsize=12)
ax.set_ylabel('Precio', fontsize=12)
ax.legend(fontsize=12)
ax.grid(True)
plt.show()

### Próximos Pasos

1.  **Entrenar Modelo de Clasificación:** Usar LightGBM para predecir la `target_signal`.
2.  **Entrenar Modelo LSTM:** Implementar una red neuronal LSTM para las tareas de regresión y clasificación para comparar su rendimiento.
3.  **Análisis Comparativo:** Crear una tabla que resuma el rendimiento de todos los modelos para decidir cuál es el mejor.